# MLOps Emotion Detection Pipeline — Group 38
**Dataset**: `dair-ai/emotion` &nbsp;|&nbsp; **Task**: 6-class Text Emotion Classification  
**Model**: `distilbert-base-uncased` fine-tuned for sequence classification  
**Tracking**: Weights & Biases &nbsp;|&nbsp; **Registry**: Hugging Face Hub

---

### Emotion Classes
| ID | Label    |
|----|----------|
| 0  | sadness  |
| 1  | joy      |
| 2  | love     |
| 3  | anger    |
| 4  | fear     |
| 5  | surprise |

> **Note**: `mlops-group-project.ipynb` in this repo is a reference notebook for pipeline structure only.  
> This notebook (`emotion-detection-kaggle.ipynb`) is the actual training notebook for **Group 38**.

## Step 0 — Load Secrets
**On Kaggle**: Add `WANDB_API_KEY`, `HF_TOKEN`, `HF_REPO_ID` in  
*Add-ons → Secrets* before running.

In [ ]:
import os

# ── Kaggle Secrets ──────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = _secrets.get_secret("WANDB_API_KEY")
    os.environ["HF_TOKEN"]      = _secrets.get_secret("HF_TOKEN")
    os.environ["HF_REPO_ID"]    = _secrets.get_secret("HF_REPO_ID")
    print("Kaggle secrets loaded.")
except Exception:
    print("Not on Kaggle — reading secrets from environment variables.")

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
HF_TOKEN      = os.environ.get("HF_TOKEN", "")
HF_REPO_ID    = os.environ.get("HF_REPO_ID", "your-username/mlops-group38-emotion")

print(f"HF_REPO_ID : {HF_REPO_ID}")
print(f"WANDB key  : {'set' if WANDB_API_KEY else 'NOT SET'}")
print(f"HF token   : {'set' if HF_TOKEN else 'NOT SET'}")

## Step 1 — Install Dependencies

In [ ]:
%%capture
# Upgrade transformers + peft together to fix EncoderDecoderCache incompatibility
!pip install -q --upgrade \
    "transformers==4.46.3" \
    "peft>=0.13.0" \
    datasets \
    "accelerate>=0.34.0" \
    wandb \
    scikit-learn \
    evaluate \
    seaborn \
    matplotlib \
    huggingface_hub
print("Install done — use Session ▸ Restart & Run All to reload packages")

## Step 2 — Initialise W&B

In [ ]:
import wandb

WANDB_PROJECT  = "emotion-mlops-group38-project"
WANDB_RUN_NAME = "run-v1"   # change to 'run-v2' for second experiment

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY, relogin=False)
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            "model": "distilbert-base-uncased",
            "dataset": "dair-ai/emotion",
            "task": "emotion-detection",
            "num_labels": 6,
            "train_size": 16000,
            "val_size": 2000,
            "test_size": 2000,
            "max_seq_length": 128,
            "learning_rate": 2e-5,
            "epochs": 4,
            "batch_size": 32,
            "weight_decay": 0.01,
            "warmup_ratio": 0.06,
        },
        reinit="finish_previous",   # replaces deprecated reinit=True
    )
    print("W&B run started:", wandb.run.url)
else:
    print("W&B disabled — WANDB_API_KEY not set.")

## Step 3 — Configuration

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # suppress fork warning

# ── Hyperparameters ──────────────────────────────────────────────────────────
MODEL_NAME              = "distilbert-base-uncased"
DATASET_NAME            = "dair-ai/emotion"
MAX_SEQ_LENGTH          = 128
TRAIN_SIZE              = 16000
VAL_SIZE                = 2000
TEST_SIZE               = 2000
LEARNING_RATE           = 2e-5
NUM_TRAIN_EPOCHS        = 4
TRAIN_BATCH_SIZE        = 32
EVAL_BATCH_SIZE         = 64
WEIGHT_DECAY            = 0.01
WARMUP_RATIO            = 0.06
OUTPUT_DIR              = "/kaggle/working/checkpoints"

# ── Label mapping ────────────────────────────────────────────────────────────
ID2LABEL = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

print(f"Model      : {MODEL_NAME}")
print(f"Dataset    : {DATASET_NAME}")
print(f"Num labels : {NUM_LABELS}  →  {list(ID2LABEL.values())}")

## Step 4 — Load & Explore Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

raw = load_dataset(DATASET_NAME)
print(raw)

# Show sample rows
df_sample = pd.DataFrame(raw["train"].select(range(10)))
df_sample["emotion"] = df_sample["label"].map(ID2LABEL)
display(df_sample)

In [ ]:
# ── Class distribution ───────────────────────────────────────────────────────
from collections import Counter

for split_name in ["train", "validation", "test"]:
    counts = Counter(raw[split_name]["label"])
    print(f"\n{split_name.upper()} ({len(raw[split_name])} samples):")
    for label_id, cnt in sorted(counts.items()):
        print(f"  {ID2LABEL[label_id]:10s}: {cnt:5d}  ({cnt/len(raw[split_name])*100:.1f}%)")

# Bar chart
train_counts = Counter(raw["train"]["label"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([ID2LABEL[i] for i in sorted(train_counts)],
       [train_counts[i] for i in sorted(train_counts)],
       color=sns.color_palette("husl", 6))
ax.set_title("Training Set Class Distribution — dair-ai/emotion")
ax.set_xlabel("Emotion")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Step 5 — Data Cleaning & Tokenisation

In [ ]:
import re

def clean_text(text: str) -> str:
    """Normalise raw text for emotion classification.

    Cleaning decisions:
      1. Lower-case — emotion signals are not case-dependent
      2. Remove URLs — carry no emotion signal
      3. Remove @mentions — not relevant to emotion content
      4. Strip # symbol but keep hashtag words
      5. Remove non-ASCII characters — keeps text clean for tokeniser
      6. Keep sentiment-bearing punctuation (. , ! ?)
      7. Collapse multiple whitespace
    """
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)      # remove URLs
    text = re.sub(r"@\w+", "", text)                          # remove @mentions
    text = re.sub(r"#(\w+)", r"\1", text)                    # strip # symbol
    text = text.encode("ascii", errors="ignore").decode()     # remove non-ASCII
    text = re.sub(r"[^a-z0-9\s'.,!?]", " ", text)           # keep useful chars
    text = re.sub(r"\s+", " ", text).strip()                 # collapse spaces
    return text

# Quick sanity check
samples = [
    "I feel SO happy today!!! 😊 #blessed",
    "This is terrible @user https://example.com",
    "I'm afraid of what might happen...",
]
for s in samples:
    print(f"  IN : {s}")
    print(f"  OUT: {clean_text(s)}\n")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)

def preprocess(batch):
    cleaned = [clean_text(t) for t in batch["text"]]
    enc = tokenizer(cleaned, truncation=True, padding="max_length", max_length=MAX_SEQ_LENGTH)
    enc["labels"] = batch["label"]
    return enc

train_raw = raw["train"].select(range(min(TRAIN_SIZE, len(raw["train"]))))
val_raw   = raw["validation"].select(range(min(VAL_SIZE, len(raw["validation"]))))
test_raw  = raw["test"].select(range(min(TEST_SIZE, len(raw["test"]))))

# Apply cleaning + tokenisation
remove_cols = ["text", "label"]
train_ds = train_raw.map(preprocess, batched=True, remove_columns=remove_cols)
val_ds   = val_raw.map(preprocess,   batched=True, remove_columns=remove_cols)
test_ds  = test_raw.map(preprocess,  batched=True, remove_columns=remove_cols)

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print("Sample keys:", list(train_ds[0].keys()))

## Step 6 — Load Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
    token=HF_TOKEN or None,
)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

## Step 7 — Define Metrics

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

LABEL_NAMES = [ID2LABEL[i] for i in range(NUM_LABELS)]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    report = classification_report(
        labels, preds,
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )
    return {
        "accuracy":        report["accuracy"],
        "f1":              report["weighted avg"]["f1-score"],
        "precision":       report["weighted avg"]["precision"],
        "recall":          report["weighted avg"]["recall"],
        "macro_f1":        report["macro avg"]["f1-score"],
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall":    report["macro avg"]["recall"],
    }

print("compute_metrics function ready.")

## Step 8 — Train

**Experiment V1** (default): `lr=2e-5`, `epochs=4`  
**Experiment V2**: change `LEARNING_RATE = 5e-5`, `NUM_TRAIN_EPOCHS = 6`, `WANDB_RUN_NAME = 'run-v2'`

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer)
report_to     = "wandb" if WANDB_API_KEY else "none"

training_args = TrainingArguments(
    output_dir              = OUTPUT_DIR,
    num_train_epochs        = NUM_TRAIN_EPOCHS,
    per_device_train_batch_size = TRAIN_BATCH_SIZE,
    per_device_eval_batch_size  = EVAL_BATCH_SIZE,
    learning_rate           = LEARNING_RATE,
    weight_decay            = WEIGHT_DECAY,
    warmup_ratio            = WARMUP_RATIO,
    eval_strategy           = "epoch",
    save_strategy           = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1",
    logging_steps           = 100,
    fp16                    = True,   # GPU only
    report_to               = report_to,
    run_name                = WANDB_RUN_NAME,
    push_to_hub             = False,
)

trainer = Trainer(
    model             = model,
    args              = training_args,
    train_dataset     = train_ds,
    eval_dataset      = val_ds,
    processing_class  = tokenizer,   # replaces deprecated tokenizer= param
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,
)

trainer.train()

## Step 9 — Evaluate on Test Split

In [ ]:
import json
import seaborn as sns
from sklearn.metrics import confusion_matrix

pred_output = trainer.predict(test_ds)
preds  = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

# Classification report
report_txt  = classification_report(labels, preds, target_names=LABEL_NAMES, zero_division=0)
report_dict = classification_report(labels, preds, target_names=LABEL_NAMES,
                                    output_dict=True, zero_division=0)
print(report_txt)

# Save to files
import os
os.makedirs("/kaggle/working/eval_results", exist_ok=True)
with open("/kaggle/working/eval_results/classification_report.txt", "w") as f:
    f.write(report_txt)
with open("/kaggle/working/eval_results/classification_report.json", "w") as f:
    json.dump(report_dict, f, indent=2)

# Confusion matrix
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            cmap="Blues", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Emotion Detection (Test Split)")
plt.tight_layout()
plt.savefig("/kaggle/working/eval_results/confusion_matrix.png", dpi=150)
plt.show()

# Log to W&B
if wandb.run is not None:
    wandb.log({"confusion_matrix": wandb.Image("/kaggle/working/eval_results/confusion_matrix.png")})
    wandb.log({
        "final/accuracy":        report_dict["accuracy"],
        "final/weighted_f1":     report_dict["weighted avg"]["f1-score"],
        "final/macro_f1":        report_dict["macro avg"]["f1-score"],
        "final/macro_precision": report_dict["macro avg"]["precision"],
        "final/macro_recall":    report_dict["macro avg"]["recall"],
    })

## Step 10 — Push to Hugging Face Hub

In [ ]:
if HF_TOKEN:
    print(f"Pushing model to Hub: {HF_REPO_ID}")
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    hub_url = f"https://huggingface.co/{HF_REPO_ID}"
    print("Model URL:", hub_url)
    if wandb.run is not None:
        wandb.run.summary["huggingface_model"] = hub_url
else:
    print("HF_TOKEN not set — skipping Hub push.")

## Step 11 — Inference Demo

In [ ]:
from transformers import pipeline as hf_pipeline

classifier = hf_pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None,
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
)

test_sentences = [
    "I am so happy today, everything feels perfect!",
    "I feel really sad and lonely right now.",
    "This makes me so angry, how could they do this?",
    "I am terrified of what will happen next.",
    "Wow, I did not expect that at all!",
    "I love spending time with my family.",
]

print(f"{'Text':<55} {'Predicted Emotion':<12} {'Confidence'}")
print("-" * 80)
for sentence in test_sentences:
    result = classifier(sentence)[0]
    best   = sorted(result, key=lambda x: x["score"], reverse=True)[0]
    print(f"{sentence[:54]:<55} {best['label']:<12} {best['score']:.4f}")

## Step 12 — Finish W&B Run

In [ ]:
if wandb.run is not None:
    wandb.finish()
    print("W&B run finished.")
print("All done!")